# MLflow

MLflow conserve les paramètres, les métriques, les artifacts et le modèle de chaque entraînement.
Ce notebook journalise un LightGBM en local, puis envoie les artifacts vers un serveur.
Prérequis cloud : VM Debian 12 de type `g1-small`, bucket Cloud Storage, clé JSON d'un compte de service (rôles Créateur et Lecteur des objets de ce bucket).
Ne pas committer cette clé. Ne pas lancer `mlflow server` dans le notebook : le démarrer dans un terminal.


## Suivi local

Installer MLflow dans un terminal (`pip install mlflow`), puis lancer le serveur hors du notebook.
L'interface locale écoute sur `http://127.0.0.1:5000`.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

import mlflow
import mlflow.sklearn

from lightgbm.sklearn import LGBMClassifier
from mlflow.models import infer_signature
from sklearn.metrics import f1_score, PrecisionRecallDisplay, precision_recall_curve

X_train = pd.read_csv(os.path.expanduser("data/X_train.csv"))
X_test = pd.read_csv(os.path.expanduser("data/X_test.csv"))
y_train = pd.read_csv(os.path.expanduser("data/y_train.csv")).values.flatten()
y_test = pd.read_csv(os.path.expanduser("data/y_test.csv")).values.flatten()


In [ ]:
int_columns = ['product_id', 'brand', 'num_views_session', 'num_views_product',
               'category', 'sub_category', 'hour', 'minute', 'weekday',
               'duration', 'num_prev_sessions', 'num_prev_product_views']
X_train[int_columns] = X_train[int_columns].astype('float64')
X_test[int_columns] = X_test[int_columns].astype('float64')
hyp_params = {
    "num_leaves": 60,
    "min_child_samples": 10,
    "max_depth": 12,
    "n_estimators": 100,
    "learning_rate": 0.1
}


In [ ]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")

mlflow.set_experiment("purchase_predict")

with mlflow.start_run() as run:
    model = LGBMClassifier(**hyp_params, objective="binary", verbose=-1)
    model.fit(X_train, y_train)

    score = f1_score(y_test, model.predict(X_test))

    mlflow.log_params(hyp_params)
    mlflow.log_metric("f1", score)

    print(mlflow.get_artifact_uri())

    signature = infer_signature(X_train, model.predict(X_train))
    input_example = X_test.iloc[0:1].copy()

    mlflow.sklearn.log_model(model, "model", signature=signature, input_example=input_example)


In [ ]:
from mlflow.tracking import MlflowClient

client = MlflowClient(
    tracking_uri="http://127.0.0.1:5000"
)

client.get_metric_history(run.info.run_id, key='f1')


## Plusieurs runs

`train_model` entraîne le modèle, enregistre la courbe précision-rappel et journalise le run.


In [ ]:
def save_pr_curve(X, y, model):
    plt.figure(figsize=(16,11))
    prec, recall, _ = precision_recall_curve(y, model.predict_proba(X)[:,1], pos_label=1)
    pr_display = PrecisionRecallDisplay(precision=prec, recall=recall).plot(ax=plt.gca())
    plt.title("PR Curve", fontsize=16)
    plt.gca().xaxis.set_major_formatter(mtick.PercentFormatter(1, 0))
    plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(1, 0))
    plt.savefig(os.path.expanduser("data/pr_curve.png"))
    plt.close()

def train_model(params):

    with mlflow.start_run() as run:
        model = LGBMClassifier(**params, objective="binary", verbose=-1)
        model.fit(X_train, y_train)

        score = f1_score(y_test, model.predict(X_test))
        save_pr_curve(X_test, y_test, model)

        mlflow.log_params(hyp_params)
        mlflow.log_metric("f1", score)
        mlflow.log_artifact(os.path.expanduser("data/pr_curve.png"), artifact_path="plots")

        signature = infer_signature(X_train, model.predict(X_train))
        input_example = X_test.iloc[0:1].copy()

        mlflow.sklearn.log_model(model, "model", signature=signature, input_example=input_example)


In [ ]:
train_model({**hyp_params, **{'n_estimators': 200, 'learning_rate': 0.05}})
train_model({**hyp_params, **{'n_estimators': 500, 'learning_rate': 0.025}})
train_model({**hyp_params, **{'n_estimators': 1000, 'learning_rate': 0.01}})


## Serveur MLflow

Sur la VM Debian 12, installer MLflow et le client Cloud Storage.
Les artifacts distants vont dans un bucket. Le compte de service de la VM y a déjà accès.
Créer un compte de service limité à ce bucket, avec les rôles Créateur des objets de l'espace de stockage et Lecteur des objets de l'espace de stockage, puis télécharger sa clé JSON.
Remplacer `gs://<votre_bucket_ici>/mlflow` dans l'unité systemd. Le service redémarre avec la VM.


Enregistrer l'unité dans `/etc/systemd/system/mlflow.service`, puis l'activer.


L'interface est disponible sur l'adresse IP externe de la VM, port 80.
Coller la clé JSON du compte de service à la place du commentaire. Ne pas y mettre d'autre secret, et ne pas committer le fichier.


In [ ]:
%%writefile data/mlflow-key.json
# Insérez votre clé ici


In [ ]:
import os
from google.cloud import storage

# Authentification à Google Cloud avec la clé correspondant au compte de service MLflow
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = os.path.expanduser("data/mlflow-key.json")

# Nouvel URI de l'interface MLflow
mlflow.set_tracking_uri("http://35.184.99.125/") # Mettez l'adresse de Google Compute Engine ici
client = storage.Client()


In [ ]:
mlflow.set_experiment("purchase_predict")

train_model({**hyp_params, **{'n_estimators': 200, 'learning_rate': 0.05}})
